<a href="https://colab.research.google.com/github/davejeon/dsr_agentic_workflow/blob/main/Deep_Agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install packages
%pip install langchain-google-genai
%pip install deepagents tavily-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.9/197.9 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.3/114.3 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 399.6/399.6 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 763.1/763.1 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.3/234.3 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 2.6 MB/s eta 0:00:00
  Attempting uninstall: langsmith
    Found existing installation: langsmith 0.7.34
    Uninstalling langsmith-0.7.34:
      Successfully uninstalled langsmith-0.7.34
  Attempting uninstall: langchain-cor

In [2]:
# Import packages
from langchain.messages import AIMessage
from langchain_core.output_parsers import PydanticOutputParser
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.structured_output import ToolStrategy
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, AIMessage, SystemMessage
import os
from google.colab import userdata
from langchain.tools import tool


In [4]:
# Connect notebook to langsmith
os.environ['LANGSMITH_TRACING'] = userdata.get('LANGSMITH_TRACING')
os.environ['LANGSMITH_ENDPOINT'] = userdata.get('LANGSMITH_ENDPOINT')
os.environ['LANGSMITH_API_KEY'] = userdata.get('LANGSMITH_API_KEY')
os.environ['LANGSMITH_PROJECT'] = userdata.get('LANGSMITH_PROJECT')
os.environ['GOOGLE_API_KEY'] = userdata.get('GEMINI_API_KEY')
os.environ['TAVILY_API_KEY'] = userdata.get('TAVILY')
# Test the output
userdata.get('LANGSMITH_PROJECT')

'dsr_agentic_ai'

In [5]:
# Create a search tool
import os
from typing import Literal

from tavily import TavilyClient
from deepagents import create_deep_agent

tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])


def internet_search(
    query: str,
    max_results: int = 5,
    topic: Literal["general", "news", "finance"] = "general",
    include_raw_content: bool = False,
):
    """Run a web search"""
    return tavily_client.search(
        query,
        max_results=max_results,
        include_raw_content=include_raw_content,
        topic=topic,
    )

In [6]:
# System prompt to steer the agent to be an expert researcher
research_instructions = """You are an expert researcher.
Your job is to conduct thorough research and then write an overview.

You have access to an internet search tool as your primary means of gathering information.

## `internet_search`

Use this to run an internet search for a given query. You can specify the max number of results to return, the topic, and whether raw content should be included.
"""

agent = create_deep_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=[internet_search],
    system_prompt=research_instructions,
)

In [9]:
stream = agent.stream_events({
    "messages": [{"role": "user", "content": "What major events happened in the news today?"}],
}, version="v3")


In [10]:
for message in stream.messages:
    for delta in message.text:
        print(delta, end="", flush=True)

final_state = stream.output

{"query": "major news headlines today", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://markets.businessinsider.com/news/stocks/crypto-news-today-alphapepe-releases-major-alphaswap-ai-dex-update-while-bitcoin-price-prediction-targets-150-000-1036145834", "title": "Crypto News Today: AlphaPepe Releases Major AlphaSwap AI DEX Update While Bitcoin Price Prediction Targets $150,000 - markets.businessinsider.com", "score": 0.56719416, "published_date": "Wed, 13 May 2026 00:36:40 GMT", "content": "# Crypto News Today: AlphaPepe Releases Major AlphaSwap AI DEX Update While Bitcoin Price Prediction Targets $150,000. MONACO, May 12, 2026 (GLOBE NEWSWIRE) -- Crypto news today is turning toward AlphaPepe after the project released a major update to AlphaSwap, its AI-powered decentralized exchange designed for safer and smarter on-chain trading. The update arrives as Stage 16 continues at $0.01683 per token, the presale approaches $1.2 million raised, the ho